<!-- cabecera-entorno -->
## Antes de empezar

**Clase 13 · Estadística inferencial** — Bloque 3 · Reto. Este cuaderno lo recorre **usted solo**,
leyendo: cada tarea trae la explicación y los comandos que necesita. El profesor circula por el salón
resolviendo dudas. Es el entregable de la clase.

**La rutina de siempre:** `git pull` antes de clase, y el entorno virtual activo (`(.venv)` en la
terminal). Si va a modificar este archivo, trabaje sobre una copia: duplique `reto.ipynb` como
`reto_mio.ipynb` y edite el duplicado. Así `git pull` nunca le reclama.

**Si la celda de abajo falla, no siga:** la respuesta está en el manual del entorno,
[`../INSTALACION.md`](../INSTALACION.md).

| Si ve esto | Qué pasó | Dónde se arregla |
|------------|----------|------------------|
| `ModuleNotFoundError` | El entorno virtual no está activo, o VSCode eligió otro intérprete | Manual, secciones 6.3 y 8.4, y problema 5 |
| `FileNotFoundError` al leer el CSV | El notebook se abrió desde otra carpeta, o falta hacer `git pull` | Manual, problema 6 |
| El kernel no aparece en VSCode | Falta la extensión Jupyter o `ipykernel` dentro del entorno | Manual, problema 4 |

In [ ]:
# Verificación del entorno. Si algo falla aquí, la solución está en ../INSTALACION.md
import sys
from pathlib import Path

try:
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    from scipy import stats
except ModuleNotFoundError as error:
    raise ModuleNotFoundError(
        f"Falta la librería '{error.name}'. Active el entorno virtual y seleccione el intérprete "
        ".venv en VSCode (Ctrl+Shift+P > Python: Select Interpreter), luego reinicie el kernel. "
        "Ver ../INSTALACION.md, problema 5."
    ) from error

print("Intérprete:", sys.executable)
if ".venv" not in sys.executable:
    print("AVISO: este no parece el Python del entorno virtual. En VSCode: Ctrl+Shift+P >",
          "'Python: Select Interpreter' > el que dice .venv, y reinicie el kernel.")

RUTA_VERIFICACION = "../datos/educacion_estadisticas.csv"
if Path(RUTA_VERIFICACION).exists():
    print("Datos: encontrados en", RUTA_VERIFICACION)
else:
    print("FALTA el archivo", RUTA_VERIFICACION, "- abra en VSCode la carpeta raíz del curso",
          "y ejecute 'git pull'. Ver ../INSTALACION.md, problema 6.")

# Clase 13 · Reto — Inferencia sobre la educación en Colombia

**Nombre:**

**Fecha:**

**Dataset:** `../datos/educacion_estadisticas.csv` — Ministerio de Educación Nacional, vía
datos.gov.co. **Consigna completa:** `reto.md`

Usted es analista del Ministerio de Educación. Le entregan la serie histórica de indicadores educativos
por departamento y le hacen tres preguntas:

1. ¿Qué podemos afirmar sobre los indicadores nacionales, y con cuánta seguridad?
2. ¿Los departamentos urbanizados y los rurales dispersos tienen resultados académicos distintos?
3. ¿La cobertura educativa cambió entre la década pasada y la actual?

Su trabajo **no** es responder "sí" o "no". Es responder **con cuánta certeza** y **de cuánto tamaño**,
y escribirlo de forma que un funcionario que no sabe estadística pueda decidir con eso.

### Es el mismo archivo del demo, y son otras variables

La regla del curso es que el reto usa datos que no aparecieron en el demo. **Hoy la excepción es
explícita y está declarada:** el archivo es el mismo, las variables no.

| | Demo | Reto |
|--|------|------|
| Variables de los intervalos | `tasa_matriculacion_5_16` | `cobertura_neta`, `aprobacion`, `repitencia` |
| Variables de las pruebas | `desercion` | `aprobacion` y `cobertura_bruta` |
| Comparaciones | Urbanizado contra rural disperso | Urbanizado contra rural disperso **y** 2011-2018 contra 2019-2024 |

**Por qué.** La técnica de hoy es corta de teclear y larga de interpretar. Un archivo nuevo se llevaría
15 de los 60 minutos en limpieza y reconocimiento de un dominio desconocido, que es justo lo que ya se
evaluó en el Momento 1. Manteniendo el archivo, los 60 minutos se gastan en lo que se evalúa hoy:
plantear, calcular y **redactar**. Ninguna de las cinco variables de este cuaderno aparece en el demo, y
la comparación temporal tampoco.

### Las columnas que va a usar

| Columna | Qué mide | Tipo |
|---------|----------|------|
| `ano` | Año del registro, 2011 a 2024 | numérica |
| `c_digo_departamento` | Código DANE del departamento. Está limpio | numérica (es un código, no una cantidad) |
| `cobertura_neta` | % de la población en edad escolar matriculada en el nivel que le corresponde | numérica |
| `cobertura_bruta` | % de matriculados sobre la población en edad escolar, sin importar el nivel. **Puede pasar de 100** | numérica |
| `aprobacion` | % de estudiantes que aprueban el año | numérica |
| `repitencia` | % de estudiantes que repiten el año | numérica |

### Tres advertencias antes de escribir la primera línea

1. **`dropna()` antes de calcular. Siempre.** Las columnas de este archivo tienen faltantes dispersos
   (`cobertura_neta` 40, `aprobacion` 55, `repitencia` 0). Sin limpiar, `stats.sem` devuelve `nan` y el
   intervalo sale `(nan, nan)` **sin lanzar ningún error**. Es el atasco número uno de esta clase.
2. **El `n` de cada cálculo se reporta.** Si el `n` no coincide con las filas del archivo, eso no está
   mal: está bien reportado. Lo que está mal es no decirlo.
3. **`departamento` no se usa como llave.** El nombre viene sucio (`'  Nariño  '`, `'antioquia'`,
   `'Guainia'` y `'Guainía'`, `'BOGOTA, D,C,'`). El paso 0 lo estandariza con la receta de la clase 3,
   pero la llave sigue siendo `c_digo_departamento`: un código no tiene ortografía.

## Cómo se recorre este cuaderno

Usted trabaja solo. Nadie va a dictar los pasos desde el tablero, así que cada tarea trae todo lo que
necesita para resolverse leyendo:

| Parte de la tarea | Qué contiene |
|-------------------|--------------|
| **La pregunta** | Lo que hay que responder, escrito en español |
| **El concepto** | Qué técnica aplica y por qué esa y no otra |
| **Los comandos** | Las instrucciones exactas que va a usar, escritas de forma genérica |
| **Lo que decide usted** | Qué columna, qué orden, qué texto. Ahí no hay respuesta escrita |
| **La celda de código** | Los pasos numerados en comentarios. Usted escribe las líneas |
| **La comprobación** | `comprobar('TN', ...)` le dice si el resultado es el correcto, sin mostrárselo |

**Por qué esto sigue siendo un reto y no una copia.** El demo trabajó sobre la matriculación y la
deserción. Aquí hay cinco variables que no vio, una comparación temporal que no se hizo, y **una de las
dos pruebas va a dar un resultado que usted no esperaba**. La técnica se guía; el criterio y la
redacción no, y son lo que se evalúa.

**Las ocho tareas (T1 a T8)** están repartidas en tres partes. Las de gráfico se comprueban distinto: no
hay una única respuesta correcta para un gráfico, así que lo que se revisa es que esté dibujado,
titulado, con los dos ejes etiquetados y —en el de la prueba— con la anotación del resultado escrita
encima. Es exactamente lo que pide la rúbrica.

**La parte C es la que más pesa** y es la única donde no se le da el orden de los comandos.

> **Advertencia que se cumple en serio.** Una de las dos pruebas va a dar **no significativa**. Cuando
> pase, **no cambie el corte de años, ni la columna, ni los grupos** buscando que baje de 0,05. Eso se
> llama **p-hacking** y aquí le pone techo a la dimensión Ser. Un resultado no significativo bien
> reportado vale más que uno significativo fabricado.

---

## Paso 0 · Preparación

**El concepto.** Cuatro librerías: **pandas** manipula la tabla, **numpy** hace la aritmética,
**matplotlib** dibuja y **scipy.stats** trae las funciones estadísticas de la clase de hoy.

Las tres celdas de abajo ya están escritas: son la misma limpieza del demo más la columna `periodo`,
que es nueva y la va a necesitar en la prueba B. Ejecútelas y lea lo que imprimen.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

pd.set_option('display.max_columns', 40)
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

print('pandas', pd.__version__, '| scipy', __import__('scipy').__version__)

In [ ]:
# Este cuaderno vive en clase13/reto/, y el CSV dos carpetas más arriba, en datasets/
df = pd.read_csv('../datos/educacion_estadisticas.csv')
print('Archivo crudo:', df.shape)

# 0. Nombre del departamento: espacios, mayúsculas y tildes, la receta de la clase 3.
#    Sin esto, 'Guainia' y 'Guainía' son dos departamentos distintos para pandas.
df['departamento'] = (df['departamento'].astype(str)
                      .str.strip()
                      .str.normalize('NFKD')
                      .str.encode('ascii', 'ignore')
                      .str.decode('utf-8')
                      .str.upper())

# 1. Año a entero (el archivo mezcla 2018.0 con 2018)
df['ano'] = df['ano'].astype(int)

# 2. Quitar los 20 pares (año, departamento) repetidos
df = df.drop_duplicates(subset=['ano', 'c_digo_departamento']).reset_index(drop=True)

# 3. Grupo territorial: proxy construido por nosotros a partir del código DANE.
#    NO es una columna del dataset. Hay que declararlo cada vez que se reporte.
CODIGOS_URBANIZADO = [11, 5, 76, 8, 68, 25, 66, 17, 63, 13, 54]
CODIGOS_RURAL_DISPERSO = [81, 85, 86, 91, 94, 95, 97, 99, 27]

df['grupo_territorial'] = np.where(
    df['c_digo_departamento'].isin(CODIGOS_URBANIZADO), 'Urbanizado',
    np.where(df['c_digo_departamento'].isin(CODIGOS_RURAL_DISPERSO), 'Rural disperso', 'Otro')
)

# 4. Periodo: la variable de corte de la prueba B. También es una decisión nuestra.
df['periodo'] = np.where(df['ano'] <= 2018, '2011-2018', '2019-2024')

print('Después de limpiar:', df.shape)
print()
print(df['grupo_territorial'].value_counts().to_string())
print()
print(df['periodo'].value_counts().to_string())

In [ ]:
# Cuántos faltantes tiene cada columna que va a usar. Mírelos ANTES de calcular.
for columna in ['cobertura_neta', 'aprobacion', 'repitencia', 'cobertura_bruta']:
    faltantes = df[columna].isna().sum()
    print(f'{columna:18s} n con dato = {df[columna].notna().sum():3d}   faltantes = {faltantes}')

### El verificador

La celda de abajo define `comprobar(...)` y `comprobar_grafico(...)`. Ejecútela una vez y siga adelante:
es andamiaje del curso, no materia de la clase.

In [ ]:
# Verificador de las ocho tareas. Ejecute esta celda una vez y siga adelante.
# No hace falta entenderla hoy: es andamiaje del curso, no materia de la clase.
import hashlib

_RESULTADOS = {}

_CLAVES = ["T1", "T2", "T3", "T4", "T5", "T6", "T7", "T8"]

_PISTAS = {
    "T1": "Dentro del bucle, la serie ya viene con dropna() aplicado. Las seis claves del diccionario van en este orden: variable, n, media, desv_est, ic_inf, ic_sup. En scale va el ERROR ESTANDAR (stats.sem), no la desviacion estandar: si el intervalo le sale de decenas de puntos de ancho, es eso. Y la desviacion se pide con ddof=1.",
    "T2": "plt.errorbar no devuelve el eje: abra la figura con fig, eje = plt.subplots() y dibuje con eje.errorbar(...). yerr es la MITAD del ancho del intervalo, o sea (ic_sup - ic_inf) / 2. Faltan el titulo o alguna etiqueta de eje si el verificador lo dice.",
    "T3": "Es la media del grupo Urbanizado menos la del grupo Rural disperso, sobre la columna de aprobacion y con dropna() aplicado a cada grupo por separado. Si le sale negativo, invirtio el orden de la resta; si le sale nan, falta el dropna().",
    "T4": "stats.ttest_ind(a, b, equal_var=False) devuelve dos cosas: el estadistico y el p-valor, en ese orden. Se recoge con t, p = stats.ttest_ind(...). Guarde el PRIMERO. El orden de los dos grupos tiene que ser el mismo de la tarea anterior, o el signo sale al reves.",
    "T5": "eje.boxplot([serie_a, serie_b], tick_labels=[...]) sobre un eje que usted abrio con fig, eje = plt.subplots(). Faltan el titulo, alguna etiqueta de eje, o la anotacion escrita sobre el grafico con eje.text(x, y, '...').",
    "T6": "Es la media del periodo 2019-2024 menos la del periodo 2011-2018, sobre cobertura_bruta y con dropna() en cada grupo. La columna 'periodo' ya esta creada en el paso 0: filtre con ella, no con el ano suelto.",
    "T7": "Es el SEGUNDO elemento de lo que devuelve stats.ttest_ind(..., equal_var=False), el p-valor. Con los mismos dos grupos de la tarea anterior. Si le da por debajo de 0.05, revise que este comparando cobertura_bruta y no otra columna.",
    "T8": "Cinco columnas, en este orden: n_urbano, n_rural, diferencia, ic_inf, ic_sup. La diferencia es la de la tarea 3. El error estandar de la diferencia es np.sqrt(v1/n1 + v2/n2) con varianzas de ddof=1, los grados de libertad salen de la formula de Welch del inventario, y el intervalo es stats.t.interval con esos grados en df, la diferencia en loc y ese error estandar en scale."
}

_ESPERADO = {
    "T1": "652e5146b6",
    "T3": "4479e7e9a5",
    "T4": "8898af2b1f",
    "T6": "5170493e8e",
    "T7": "157dd75359",
    "T8": "92f8131311"
}


def _firma(valor):
    """Reduce un resultado a un texto reproducible, sin importar como se calculo."""
    if isinstance(valor, pd.DataFrame):
        partes = ["DataFrame", str(valor.shape), str([str(c) for c in valor.columns]),
                  str([str(i) for i in valor.index])]
        for columna in valor.columns:
            serie = valor[columna]
            if pd.api.types.is_bool_dtype(serie) or not pd.api.types.is_numeric_dtype(serie):
                partes.append(f"{columna}:{[str(v) for v in serie.tolist()]}")
            else:
                partes.append(f"{columna}:{round(float(serie.sum()), 4)}")
        return "|".join(partes)
    if isinstance(valor, pd.Series):
        return "|".join(["Series", str(len(valor)), str([str(i) for i in valor.index]),
                         str([str(v) for v in valor.tolist()])])
    if not isinstance(valor, str):
        try:
            return f"numero|{round(float(valor), 4)}"
        except (TypeError, ValueError):
            pass
    return f"otro|{valor!r}"


def _huella(valor):
    return hashlib.sha256(_firma(valor).encode("utf-8")).hexdigest()[:10]


def _redondear(valor, decimales):
    if decimales is None or valor is None:
        return valor
    if isinstance(valor, (pd.DataFrame, pd.Series)):
        return valor.round(decimales)
    try:
        return round(float(valor), decimales)
    except (TypeError, ValueError):
        return valor


def comprobar(clave, valor, decimales=None):
    """Dice si el resultado es el correcto, sin revelar cual era."""
    _RESULTADOS[clave] = False
    if valor is None:
        print(f"[{clave}] Sin resolver todavia: la variable sigue valiendo None.")
        return
    valor = _redondear(valor, decimales)
    if isinstance(valor, pd.DataFrame):
        print(f"[{clave}] Usted produjo un DataFrame de {valor.shape[0]} filas "
              f"y {valor.shape[1]} columnas.")
    elif isinstance(valor, pd.Series):
        print(f"[{clave}] Usted produjo una Series de {len(valor)} elementos.")
    else:
        print(f"[{clave}] Usted produjo: {valor!r}")
    try:
        if float(valor) != float(valor):
            print(f"[{clave}] Eso es NaN. Casi siempre es un dropna() que falta: NaN se propaga "
                  f"por toda la operacion sin lanzar ningun error.")
    except (TypeError, ValueError):
        pass
    if _huella(valor) == _ESPERADO.get(clave):
        _RESULTADOS[clave] = True
        print(f"[{clave}] CORRECTO.")
    else:
        print(f"[{clave}] Todavia no coincide.")
        print(f"[{clave}] Pista: {_PISTAS[clave]}")


def comprobar_grafico(clave, eje, con_ejes=True, con_anotacion=False):
    """Revisa que el grafico exista y cumpla los requisitos que se califican.

    No hay una unica respuesta correcta para un grafico: lo que se comprueba es
    que este dibujado, titulado y etiquetado, que es lo que pide la rubrica.
    """
    _RESULTADOS[clave] = False
    if eje is None:
        print(f"[{clave}] Sin resolver todavia: la variable del eje sigue valiendo None.")
        print(f"[{clave}] Pista: {_PISTAS[clave]}")
        return
    if not hasattr(eje, "get_title"):
        print(f"[{clave}] Eso no es un eje de matplotlib, es un {type(eje).__name__}.")
        print(f"[{clave}] Pista: {_PISTAS[clave]}")
        return
    faltas = []
    if not eje.get_title().strip():
        faltas.append("falta el titulo: eje.set_title('...') o plt.title('...')")
    if con_ejes:
        if not eje.get_xlabel().strip():
            faltas.append("falta la etiqueta del eje x: eje.set_xlabel('...')")
        if not eje.get_ylabel().strip():
            faltas.append("falta la etiqueta del eje y: eje.set_ylabel('...')")
    if len(eje.collections) + len(eje.lines) + len(eje.patches) == 0:
        faltas.append("el eje esta vacio: el grafico no se dibujo sobre este eje")
    if con_anotacion and len(eje.texts) == 0:
        faltas.append("falta la anotacion escrita sobre el grafico: eje.text(x, y, '...'). "
                      "Un grafico de significancia sin la anotacion obliga a explicarlo de viva voz")
    if faltas:
        print(f"[{clave}] Todavia no esta completo:")
        for falta in faltas:
            print(f"[{clave}]   - {falta}")
        print(f"[{clave}] Pista: {_PISTAS[clave]}")
    else:
        _RESULTADOS[clave] = True
        print(f"[{clave}] CORRECTO: el grafico esta dibujado, titulado y etiquetado.")


def resumen_puntos_de_control():
    """Estado de las ocho tareas."""
    print("Punto de control")
    print("-" * 42)
    for clave in _CLAVES:
        estado = "correcto" if _RESULTADOS.get(clave) else "pendiente"
        print(f"  {clave}: {estado}")
    logrados = sum(1 for c in _CLAVES if _RESULTADOS.get(c))
    print("-" * 42)
    print(f"{logrados} de {len(_CLAVES)} {'correctas' if logrados != 1 else 'correcta'}.")


print("Verificador listo. Las tareas se comprueban con comprobar('T1', su_variable).")

---

# Parte A · Tres intervalos de confianza al 95%

**Qué se practica aquí.** Pasar de un promedio a una afirmación con incertidumbre cuantificada.

**El concepto, en cuatro frases.** Un **intervalo de confianza** declara una zona en vez de un punto.
Se construye como `media ± t * error estándar`, donde el **error estándar** es la desviación estándar
dividida por la raíz de n: mide qué tanto se movería su media si repitiera el estudio. El **95%** es una
propiedad del procedimiento, no de este intervalo: significa que si repitiera el estudio muchas veces,
alrededor del 95% de los intervalos construidos así contendrían el valor real. **Este** intervalo o lo
contiene o no lo contiene, y eso no se sabe.

| Palanca | Efecto en el ancho |
|---------|--------------------|
| Más datos (n más grande) | Más angosto |
| Más dispersión en los datos | Más ancho |
| Más confianza pedida (99% en vez de 95%) | Más ancho |

**Y el error que no avisa:** pasarle la desviación estándar a `scale` en vez del error estándar. No
lanza ninguna excepción y devuelve un intervalo del orden de la dispersión de los datos. Si su intervalo
para un porcentaje sale de 20 puntos de ancho, es eso.

### Tarea 1 · La tabla de los tres intervalos

**La pregunta.** ¿Entre qué valores está, con 95% de confianza, la media nacional de cobertura neta, de
aprobación y de repitencia?

**El concepto.** Las tres se calculan igual, así que van en un bucle y el resultado se acumula en una
tabla. Que quede en una tabla no es cosmética: la Parte C la va a graficar, y una tabla con `n`, media e
intervalo es exactamente lo que se pega en un informe.

**Los comandos.**

```python
serie = df['columna'].dropna()          # primero, siempre
n = len(serie)
media = serie.mean()
desv = serie.std(ddof=1)                # ddof=1: desviacion estandar muestral
error_estandar = stats.sem(serie)       # = desv / np.sqrt(n)
ic = stats.t.interval(0.95, df=n - 1, loc=media, scale=error_estandar)
# ic es una tupla: ic[0] es el limite inferior, ic[1] el superior
filas.append({'clave': valor, ...})     # se acumula un diccionario por variable
```

**Lo que decide usted.** Nada de la fórmula, y todo lo demás: qué va en `scale`, qué `n` reporta (¿el
del archivo o el de la serie limpia?) y en qué orden quedan las columnas. Las claves del diccionario
tienen que ser exactamente estas seis, en este orden: `variable`, `n`, `media`, `desv_est`, `ic_inf`,
`ic_sup`.

**Ojo con `ano`:** el argumento se llama `df` y son los **grados de libertad**, no el DataFrame. Es una
colisión de nombres desafortunada de scipy que confunde a todo el mundo la primera vez.

In [ ]:
# TU CÓDIGO AQUÍ
VARIABLES_IC = ['cobertura_neta', 'aprobacion', 'repitencia']

filas = []
for variable in VARIABLES_IC:
    serie = df[variable].dropna()

    # 1. n, media y desviación estándar (ddof=1) de la serie
    # 2. error estándar con stats.sem
    # 3. intervalo con stats.t.interval(0.95, df=n-1, loc=..., scale=...)
    # 4. añada a 'filas' un diccionario con las seis claves, en este orden:
    #    variable, n, media, desv_est, ic_inf, ic_sup
    pass

tabla_ic = pd.DataFrame(filas)
tabla_ic

In [ ]:
comprobar('T1', tabla_ic, decimales=3)

### Las tres interpretaciones

Una frase por intervalo, escrita **correctamente**. Recuerde: el 95% describe la tasa de acierto del
método, no la certeza sobre este intervalo en particular. Si su frase empieza por "hay un 95% de
probabilidad de que...", está mal, y en la rúbrica le pone techo a la dimensión Saber.

Incluya el `n` en cada frase. No es adorno: un intervalo sin su `n` no se puede juzgar.

**Cobertura neta:**

*Tu respuesta:*

**Aprobación:**

*Tu respuesta:*

**Repitencia:**

*Tu respuesta:*

---

**Pregunta extra.** Los tres intervalos no tienen el mismo ancho ni el mismo `n`. Mire la tabla y
explique **cuál de los tres es el más ancho y por qué**, usando las palancas de arriba. Hay dos
candidatas y solo una lo explica.

*Tu respuesta:*

### Tarea 2 · El gráfico de barras de error

**La pregunta.** ¿Cómo se ven, juntos, los tres intervalos?

**El concepto.** Un **gráfico de barras de error** dibuja la estimación puntual como un punto y la
incertidumbre como una barra vertical alrededor. Es la forma estándar de mostrar un intervalo, y hace
visible lo que una tabla esconde: qué tan distinta es la precisión de una estimación a la de otra.

**El detalle que arruina el gráfico:** `yerr` recibe **la mitad** del ancho del intervalo, porque
matplotlib dibuja esa distancia hacia arriba **y** hacia abajo. Si le pasa el ancho completo, el gráfico
sale del doble de tamaño y **miente**, sin dar ningún error.

**Los comandos.**

```python
fig, eje = plt.subplots(figsize=(8, 5))
eje.errorbar(x=lista_de_etiquetas, y=serie_de_medias, yerr=serie_de_errores,
             fmt='o', capsize=6, markersize=8)
eje.set_xlabel('...')
eje.set_ylabel('...')
eje.set_title('...')
plt.tight_layout()
plt.show()
```

**Lo que decide usted.** Qué columna de `tabla_ic` va en cada argumento, cómo se calcula `yerr` a partir
de `ic_inf` e `ic_sup`, y los tres textos. El título dice el hallazgo, no "Gráfico 1". Guarde el eje en
`eje_barras`.

In [ ]:
# TU CÓDIGO AQUÍ
# 1. Abra la figura con fig, eje = plt.subplots(...)
# 2. Calcule los errores: la MITAD del ancho de cada intervalo.
# 3. Dibuje con eje.errorbar(...), usando las columnas de tabla_ic.
# 4. Etiquete los dos ejes con su unidad y ponga un título que diga el hallazgo.

eje_barras = None

In [ ]:
comprobar_grafico('T2', eje_barras)

**Tu respuesta.** Mirando el gráfico: ¿qué barra de error es visiblemente más larga que las otras, y
qué le dice eso sobre la calidad de esa estimación? ¿Es un problema de los datos o de la realidad que
miden?

*Tu respuesta:*

---

# Parte B · Dos pruebas de hipótesis

**Qué se practica aquí.** Los cuatro pasos completos, dos veces, sobre dos preguntas distintas.

| Paso | Qué se hace |
|------|-------------|
| 1 | **Plantear** H0 y H1, en español, **antes** de calcular nada |
| 2 | **Elegir alfa.** 0,05 salvo razón para otra cosa |
| 3 | **Calcular.** Descriptivos primero, prueba después |
| 4 | **Decidir y redactar** en lenguaje de negocio |

**El concepto.** H0 es siempre la aburrida: no hay diferencia. La prueba mide qué tan raros serían sus
datos **si H0 fuera cierta**, y eso es el **p-valor**. Si serían rarísimos (p < alfa), se rechaza H0.

**Y lo que nunca se hace:** aceptar H0. Un tribunal no declara "inocente", declara "no culpable". Si el
p-valor sale grande, la frase correcta es *"no encontramos evidencia suficiente de que difieran"*, nunca
*"se demostró que son iguales"*.

`stats.ttest_ind(a, b, equal_var=False)` activa el **t-test de Welch**, que no asume varianzas iguales.
Regla del curso: se pone **siempre**. Cuando las varianzas son parecidas da lo mismo; cuando no lo son,
es el correcto y el otro está mal.

## Prueba A · ¿La aprobación difiere entre territorios?

**Paso 1 — Plantear.** Escríbalo aquí **antes** de ejecutar la celda de abajo. Una línea cada uno.

- **H0:**
- **H1:**

**Paso 2 — Alfa:**

---

### Tarea 3 · El tamaño del efecto

**La pregunta.** ¿Cuántos puntos porcentuales de aprobación separan a los departamentos urbanizados de
los rurales dispersos?

**El concepto.** **Descriptivos primero, prueba después.** El tamaño del efecto es el número que le
importa a quien decide; el p-valor solo dice si es casualidad. Reportar un p-valor sin el tamaño del
efecto es reportar "sí hay algo" sin decir de cuánto, y eso no sirve para tomar ninguna decisión.

**Los comandos.**

```python
grupo = df[df['grupo_territorial'] == 'Nombre']['columna'].dropna()
grupo.mean(), grupo.std(ddof=1), len(grupo)
```

**Lo que decide usted.** El orden de la resta. "La ventaja del urbanizado" tiene un signo, y si lo
invierte, la tabla dice lo contrario de lo que va a escribir debajo. Fije el orden ahora y manténgalo
en todas las tareas siguientes.

In [ ]:
# TU CÓDIGO AQUÍ
# 1. aprob_urbano = la aprobación del grupo 'Urbanizado', sin nulos.
# 2. aprob_rural  = la aprobación del grupo 'Rural disperso', sin nulos.
# 3. Imprima n, media y desviación estándar de cada grupo.
# 4. Guarde en diferencia_aprobacion la media del urbanizado menos la del rural.

aprob_urbano = None
aprob_rural = None
diferencia_aprobacion = None

In [ ]:
comprobar('T3', diferencia_aprobacion, decimales=4)

### Tarea 4 · El estadístico de la prueba A

**La pregunta.** Esa diferencia, ¿es más grande de lo que el azar de muestreo explicaría?

**El concepto.** El t-test devuelve dos números y hay que saber cuál es cuál. El **estadístico t** es la
diferencia medida en errores estándar: cuántas "unidades de incertidumbre" separan a los dos grupos. El
**p-valor** traduce ese t a una probabilidad bajo H0. Se recogen en ese orden.

**Los comandos.**

```python
estadistico, p = stats.ttest_ind(grupo_a, grupo_b, equal_var=False)
```

**Lo que decide usted.** El orden de los dos grupos, que tiene que ser **el mismo** de la tarea 3, o el
signo del estadístico va a contradecir su diferencia. Guarde el estadístico en `t_aprobacion` y el
p-valor en `p_aprobacion`.

In [ ]:
# TU CÓDIGO AQUÍ
# 1. Corra stats.ttest_ind sobre los dos grupos, con equal_var=False.
# 2. Guarde el estadístico en t_aprobacion y el p-valor en p_aprobacion.
# 3. Imprima los dos, y la decisión al 0.05.

t_aprobacion = None
p_aprobacion = None

In [ ]:
comprobar('T4', t_aprobacion, decimales=3)

### Tarea 5 · El boxplot de la prueba A

**La pregunta.** ¿Cómo se ve esa diferencia, dibujada?

**El concepto.** Un p-valor no se muestra en una presentación: se muestra la comparación. El boxplot
enseña las dos distribuciones completas —mediana, cuartiles, dispersión y outliers— y la anotación
encima cuenta el resultado de la prueba. Las reglas de la clase 8 siguen vigentes: **el gráfico tiene
que poder leerse solo**, sin nadie al lado explicándolo.

Tres requisitos que se comprueban: el `n` de cada grupo en la etiqueta del eje x, el eje y rotulado con
su unidad, y la **anotación escrita sobre el dibujo**.

**Los comandos.**

```python
fig, eje = plt.subplots(figsize=(8, 5))
eje.boxplot([serie_a, serie_b], tick_labels=['A (n=..)', 'B (n=..)'], patch_artist=True)
eje.set_xlabel('...')
eje.set_ylabel('...')
eje.set_title('...')

alto = max(serie_a.max(), serie_b.max())            # para colocar la anotacion arriba
eje.plot([1, 1, 2, 2], [alto + 0.5, alto + 1, alto + 1, alto + 0.5], color='black', lw=1.2)
eje.text(1.5, alto + 1.2, 'p < 0,001', ha='center')
eje.set_ylim(top=alto + 3)
plt.tight_layout()
plt.show()
```

**Lo que decide usted.** El título, que dice el **hallazgo** con su cifra y no "Boxplot 1"; y qué texto
va en la anotación, respetando la regla de formato (`p < 0,001` cuando corresponda, tres decimales si
no). Guarde el eje en `eje_box`.

In [ ]:
# TU CÓDIGO AQUÍ
# 1. Abra la figura con fig, eje = plt.subplots(...)
# 2. Boxplot de los dos grupos, con el n de cada uno en su etiqueta.
# 3. Etiquete los dos ejes y ponga un título que diga el hallazgo con su cifra.
# 4. Dibuje el corchete y escriba la anotación del resultado con eje.text(...).

eje_box = None

In [ ]:
comprobar_grafico('T5', eje_box, con_anotacion=True)

## Prueba B · ¿La cobertura bruta cambió entre periodos?

**Paso 1 — Plantear.** Otra vez, **antes** de calcular:

- **H0:**
- **H1:**

**Paso 2 — Alfa:**

**Una decisión metodológica que hay que declarar.** El corte 2011-2018 contra 2019-2024 lo pusimos
nosotros: no viene en el dataset. Es defendible y es discutible, y por eso se escribe en el reporte.

---

### Tarea 6 · El tamaño del efecto de la prueba B

**La pregunta.** ¿Cuántos puntos de cobertura bruta separan al periodo reciente del anterior?

**El concepto.** El mismo de la tarea 3, sobre otra variable de corte. La única diferencia es que aquí
los grupos los define una columna que construimos (`periodo`) en vez de una geográfica.

**Los comandos.** Los mismos de la tarea 3, cambiando la columna por la que se filtra.

**Lo que decide usted.** El orden de la resta, otra vez. Aquí la lectura natural es "cuánto cambió del
periodo viejo al nuevo", así que el reciente va primero.

In [ ]:
# TU CÓDIGO AQUÍ
# 1. cob_reciente = cobertura_bruta del periodo '2019-2024', sin nulos.
# 2. cob_anterior = cobertura_bruta del periodo '2011-2018', sin nulos.
# 3. Imprima n, media y desviación estándar de cada uno.
# 4. Guarde en diferencia_cobertura la media del reciente menos la del anterior.

cob_reciente = None
cob_anterior = None
diferencia_cobertura = None

In [ ]:
comprobar('T6', diferencia_cobertura, decimales=4)

### Tarea 7 · El p-valor de la prueba B

**La pregunta.** Ese cambio, ¿es distinguible del azar de muestreo?

**El concepto.** El mismo t-test de la tarea 4. Lo que cambia es lo que hay que hacer con el resultado,
y de eso trata la mitad de esta parte del reto.

**Los comandos.** Los mismos. Guarde el p-valor —el **segundo** de los dos números— en `p_cobertura`.

> **Lea esto antes de ejecutar.** Es probable que este p-valor no sea el que usted quería. **No cambie
> el corte de años, ni la columna, ni los grupos.** Reporte lo que salió. Un resultado no significativo
> es un resultado: significa que con estos datos no se puede distinguir el cambio del ruido, y eso es
> información útil para el Ministerio. Cambiar los grupos hasta que dé es p-hacking, y le pone techo a
> la dimensión Ser.

In [ ]:
# TU CÓDIGO AQUÍ
# 1. Corra stats.ttest_ind sobre los dos periodos, con equal_var=False.
# 2. Guarde el estadístico en t_cobertura y el p-valor en p_cobertura.
# 3. Imprima los dos, y la decisión al 0.05.

t_cobertura = None
p_cobertura = None

In [ ]:
comprobar('T7', p_cobertura, decimales=4)

### Paso 4 de las dos pruebas

Redacte las dos conclusiones con los **cuatro elementos obligatorios**, en este orden:

1. **Tamaño del efecto** en unidades del negocio (puntos porcentuales), no en unidades estadísticas.
2. **Intervalo de confianza** de ese efecto. (El de la prueba A sale en la Parte C; para la B, si no lo
   calcula, dígalo.)
3. **P-valor** y tamaños de muestra. Formato: `p < 0,001` cuando corresponda, tres decimales si no.
4. **Una frase en lenguaje llano** que un funcionario pueda leer.

**Conclusión de la prueba A:**

*Tu respuesta:*

**Conclusión de la prueba B.** Cuidado con la redacción: no se puede escribir "se demostró que no hay
diferencia" ni "las medias son iguales". Y no se puede escribir "casi significativo".

*Tu respuesta:*

---

**Pregunta de honestidad.** ¿Se le pasó por la cabeza mover el corte de años para que la prueba B
diera? Si sí, escríbalo. Nadie penaliza el impulso; lo que se penaliza es hacerlo y no decirlo. Y
explique, con la simulación del demo en mente, **por qué probar variantes hasta que una dé p < 0,05
garantiza encontrar algo aunque no haya nada**.

*Tu respuesta:*

---

# Parte C · El intervalo de la diferencia

**Es la parte que más pesa, y es la única donde no se le da el orden de los comandos.**

**El concepto.** El p-valor de la prueba A dice *"sí hay diferencia"*. No dice **de cuánto**, y quien
decide necesita el cuánto. El **intervalo de confianza de la diferencia** responde las dos preguntas de
un solo golpe: da el rango del efecto y, si no contiene el cero, dice que es significativo. Es la cifra
que cierra un informe profesional, y es la que el Momento 3 va a exigir.

Como usamos Welch, el error estándar y los grados de libertad tienen su propia fórmula. La fórmula de
los grados de libertad es fea y **no hay que memorizarla**: hay que saber que no son `n1 + n2 - 2` y
saber copiarla bien.

### El inventario de comandos

Estos son todos los comandos que necesita la tarea. Todos los ha usado ya, o están escritos aquí.
**Lo que no se le da es el orden en que se arman**, y eso es deliberado: en las sustentaciones de las
clases 6, 12 y 15 nadie le va a dar la secuencia.

```python
len(serie)
serie.var(ddof=1)
np.sqrt(x)
stats.t.interval(0.95, df=grados, loc=centro, scale=escala)
pd.DataFrame([{'clave': valor, ...}])

# error estandar de la diferencia (Welch)
np.sqrt(v1 / n1 + v2 / n2)

# grados de libertad de Welch-Satterthwaite
(v1 / n1 + v2 / n2) ** 2 / ((v1 / n1) ** 2 / (n1 - 1) + (v2 / n2) ** 2 / (n2 - 1))
```

**Lo que decide usted.** El ensamblaje completo: qué va en `loc`, qué va en `scale`, qué va en `df`, y
en qué orden se calcula cada cosa. Si se atasca, parta el problema: primero los cuatro números sueltos
(`n1`, `n2`, `v1`, `v2`), imprímalos, y solo después arme el intervalo.

### Tarea 8 · La tabla del reporte

**La pregunta.** ¿Entre qué dos valores está la ventaja de aprobación de los departamentos urbanizados?

**El formato.** Un DataFrame de **una fila** llamado `reporte_aprobacion`, con estas cinco columnas en
este orden: `n_urbano`, `n_rural`, `diferencia`, `ic_inf`, `ic_sup`.

In [ ]:
# TU CÓDIGO AQUÍ
# 1. Los cuatro números sueltos de los dos grupos de la prueba A: n y varianza (ddof=1) de cada uno.
# 2. El error estándar de la diferencia y los grados de libertad de Welch, con el inventario.
# 3. El intervalo al 95% de la diferencia.
# 4. Arme reporte_aprobacion: un DataFrame de una fila con las cinco columnas pedidas.

reporte_aprobacion = None

In [ ]:
comprobar('T8', reporte_aprobacion, decimales=3)

**Tu respuesta.** El intervalo de la diferencia, ¿contiene el cero? ¿Qué relación tiene esa respuesta
con el p-valor de la tarea 4? Diga explícitamente qué información le da el intervalo que el p-valor no
le daba.

*Tu respuesta:*

---

# Parte D · El resumen ejecutivo

De **5 a 7 frases**, dirigidas a un funcionario del Ministerio de Educación que no sabe estadística y
tiene que decidir algo con esto.

**Obligatorio:**

- Qué encontró y **de cuánto** es cada diferencia, en puntos porcentuales.
- La incertidumbre en lenguaje llano: *"con los datos disponibles, la diferencia está entre X e Y
  puntos"*.
- Decir claramente **dónde no hubo evidencia suficiente**. No esconderlo al final ni maquillarlo.
- Declarar las dos decisiones metodológicas: la agrupación urbano/rural es un proxy que construimos
  nosotros a partir del código DANE, y el corte de periodos también lo pusimos nosotros.
- Terminar con una **recomendación accionable**: algo que alguien pueda hacer el lunes.

**Prohibido:**

- Las palabras "p-valor", "hipótesis nula", "estadísticamente significativo" y "t-test". El funcionario
  no sabe qué son y no tiene por qué.
- Notación científica. `p = 1.5e-08` no se escribe en un resumen ejecutivo.
- Decir que una cosa **causa** la otra. Son datos observacionales: nadie asignó al azar los
  departamentos a ser urbanos o rurales. La frase segura es "se asocia con".
- Cualquier afirmación sin número.

*Tu respuesta:*

---

## Punto de control

Ejecute la celda de abajo para ver cuántas de las ocho tareas quedaron correctas.

Si alguna sigue pendiente, no pase de largo. Si está en el salón, levante la mano ahora, que el profesor
está aquí para eso.

In [ ]:
resumen_puntos_de_control()

---

# Parte E · Reflexión

Responda en español, dos o tres frases por pregunta. Estas no se comprueban con código: son las que se
leen en la dimensión **Ser**.

**1. El intervalo mal leído.** Un compañero escribe: *"hay un 95% de probabilidad de que la aprobación
real de Colombia esté entre 89,5% y 90,4%"*. Corrija la frase y explique en una línea qué está mal.

*Tu respuesta:*

**2. Significativo no es importante.** La prueba A dio significativa. Suponga que la diferencia hubiera
sido de 0,2 puntos porcentuales en vez de la que le dio, con el mismo p-valor. ¿Le seguiría
recomendando algo al Ministerio? ¿Por qué?

*Tu respuesta:*

**3. Del reto al Momento 3.** ¿Qué intervalo de confianza y qué prueba de hipótesis va a incluir en el
proyecto final con el dataset de su equipo? Escriba la variable, los dos grupos que compararía y la
pregunta de negocio que respondería. Si su dataset no permite ninguna comparación de dos grupos, dígalo
ahora: es mejor descubrirlo hoy que en la clase 15.

*Tu respuesta:*

---

## Opcional · Solo si terminó todo

No se comprueban y no entran en la retroalimentación.

**A. El intervalo de la diferencia de la prueba B.** El mismo ensamblaje de la tarea 8, sobre los dos
periodos. Fíjese si contiene el cero y contraste con el p-valor.

**B. La prueba no paramétrica.** `stats.mannwhitneyu(aprob_urbano, aprob_rural)` compara medianas y no
exige normalidad. ¿Cambia la conclusión? Si no cambia, eso es evidencia de que el resultado no depende
del supuesto.

**C. El d de Cohen.** La diferencia dividida por la desviación estándar combinada. Es el tamaño del
efecto **estandarizado**, o sea comparable entre variables con unidades distintas. Referencias
convencionales: 0,2 pequeño, 0,5 mediano, 0,8 grande.

```python
desv_combinada = np.sqrt((v1 + v2) / 2)
d = diferencia / desv_combinada
```

In [ ]:
# TU CÓDIGO AQUÍ (opcional)

---

## Antes de entregar

1. **Kernel → Restart and Run All.** Si algo revienta, arréglelo. Un cuaderno que no corre de arriba a
   abajo le pone techo a la dimensión Hacer.
2. Ejecute el punto de control y verifique que las ocho tareas están correctas.
3. Verifique que **todas** las celdas *Tu respuesta:* están escritas, incluidas H0 y H1 de las dos
   pruebas. El número no es el análisis.
4. Revise que ninguna frase suya diga "se acepta H0", "se demostró que no hay diferencia" ni "hay un
   95% de probabilidad de que el valor esté ahí".
5. Revise que el resumen ejecutivo no contenga "p-valor", "hipótesis nula", "estadísticamente
   significativo", "t-test" ni notación científica.
6. Revise que ningún texto suyo diga que una variable **causa** otra.
7. Guarde como `reto_clase13_APELLIDO.ipynb` y súbalo al aula virtual.

## Para el Momento 3

El proyecto final de la clase 15 exige, como mínimo, **dos intervalos de confianza y una prueba de
hipótesis** sobre el dataset de su equipo, reportados con el formato de hoy: tamaño del efecto,
intervalo, p-valor con su `n`, y conclusión en lenguaje llano.

Lo que practicó hoy es literalmente una sección de su entrega final.